# 01 - Explore MVTec data

Quick sanity check that the dataset is on disk where we expect it, and that
we can load and visualize images plus their ground-truth masks.

Run this once after you have extracted the bottle data into `data/bottle/`.

In [ ]:
# So we can import from src/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from config import DATA_DIR, CATEGORIES, DEVICE
print(f'DATA_DIR : {DATA_DIR}')
print(f'Categories: {CATEGORIES}')
print(f'Device   : {DEVICE}')

In [ ]:
# Verify the folder structure
for cat in CATEGORIES:
    cat_path = DATA_DIR / cat
    print(f'\nFolder: {cat}/')
    for sub in ['train/good', 'test', 'ground_truth']:
        p = cat_path / sub
        if not p.exists():
            print(f'  MISSING: {sub}')
            continue
        if sub == 'train/good':
            n = len(list(p.glob('*.png')))
            print(f'  {sub}/  ->  {n} images')
        else:
            subs = sorted([d.name for d in p.iterdir() if d.is_dir()])
            print(f'  {sub}/  ->  {subs}')

In [ ]:
# Plot a normal vs defective example with the matching ground-truth mask
import matplotlib.pyplot as plt
from PIL import Image

cat = CATEGORIES[0]
cat_path = DATA_DIR / cat

# Pick the first defect type
defect_types = sorted([d.name for d in (cat_path / 'test').iterdir() if d.is_dir() and d.name != 'good'])
defect = defect_types[0]

normal_img = sorted((cat_path / 'test/good').glob('*.png'))[0]
defect_img = sorted((cat_path / 'test' / defect).glob('*.png'))[0]
mask_img   = sorted((cat_path / 'ground_truth' / defect).glob('*.png'))[0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(Image.open(normal_img));     axes[0].set_title(f'{cat}: normal')
axes[1].imshow(Image.open(defect_img));     axes[1].set_title(f'{cat}: {defect}')
axes[2].imshow(Image.open(mask_img), cmap='gray'); axes[2].set_title(f'{cat}: {defect} ground-truth mask')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Sanity-check the PyTorch dataset
from dataset import build_datasets

train_ds, test_ds = build_datasets()
print(f'Train: {len(train_ds)} images')
print(f'Test : {len(test_ds)} images')

img, label = train_ds[0]
print(f'First image tensor shape: {tuple(img.shape)}')
print(f'First image label       : {label} (0=normal, 1=anomalous)')